[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/agents/google-genai/evaluator-optimizer-agent.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>

# Google GenAI SDK - 构建评估与优化智能体

## 安装依赖

In [ ]:
!pip install -q google-genai openinference-instrumentation-google-genai arize-phoenix-otel

## 连接 Arize Phoenix

In [ ]:
import os
from getpass import getpass

from phoenix.otel import register

if "PHOENIX_API_KEY" not in os.environ:
    os.environ["PHOENIX_API_KEY"] = getpass("🔑 请输入 Phoenix API 密钥： ")

if "PHOENIX_COLLECTOR_ENDPOINT" not in os.environ:
    os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = getpass("🔑 请输入 Phoenix 收集器端点： ")

tracer_provider = register(
    auto_instrument=True, project_name="google-genai-evaluator-optimizer-agent"
)
tracer = tracer_provider.get_tracer(__name__)

## 通过 Google Vertex AI 进行身份验证

In [ ]:
from google import genai

In [ ]:
!gcloud auth login

In [ ]:
# 使用 Vertex AI API 创建客户端，此处也可以改用 Google GenAI API
client = genai.Client(vertexai=True, project="<填写 GCP 项目 ID>", location="us-central1")

# 评估与优化

In [ ]:
# --- 1. 实例化模型 ---
# 全局 `genai` 配置（或此前单元格创建的 `client`）
# 已经就绪，可以直接请求模型。
story_model = "gemini-2.0-flash-001"
critic_model = "gemini-2.0-flash-001"

# --- 迭代改进配置 ---
max_iterations = 5  # 评审与修改循环的最大次数
min_quality_threshold = 8  # 质量阈值，范围 1–10


# --- 2. 生成初始短篇故事 ---
@tracer.agent()
def generate_story(story_model, client):
    initial_story = create_initial_story(story_model, client)
    return iteratively_improve_story(initial_story, story_model, critic_model, client)


@tracer.chain()
def create_initial_story(story_model, client):
    story_prompt = (
        "你是富有创意的短篇小说作家。\n"
        "请创作一个简短精彩的故事（3–4 段），描述一次意外的"
        "冒险。\n"
        "发挥想象力，同时保持简洁。\n"
        "用户输入： "
    )
    story_response = client.models.generate_content(
        model=story_model,
        contents=story_prompt + input("请输入故事提示词或创意种子： "),
    )
    current_story = story_response.text.strip()

    print("=== 初始故事 ===\n")
    print(current_story)

    return current_story


@tracer.chain()
def iteratively_improve_story(current_story, story_model, critic_model, client):
    iteration = 0
    story_quality = 0  # 初始质量分数

    while iteration < max_iterations and story_quality < min_quality_threshold:
        iteration += 1
        print(f"\n--- 迭代 {iteration} ---")

        # 生成评审意见并提取质量分数
        critique_text, story_quality = critique_story(
            current_story, critic_model, client, iteration
        )

        # 检查是否需要修改
        if story_quality >= min_quality_threshold:
            print(
                f"\n故事质量（{story_quality}/10）达到或超过阈值（{min_quality_threshold}/10)."
            )
            print("无需继续修改。")
            break

        # 根据评审意见改进故事
        current_story = revise_story(current_story, critique_text, story_model, client, iteration)

    if iteration >= max_iterations and story_quality < min_quality_threshold:
        print(
            f"\n--- 已达到最大迭代次数（{max_iterations}），仍未达到质量阈值 ---"
        )

    print("\n--- 最终故事 ---")
    print(current_story)
    return current_story


@tracer.chain()
def critique_story(current_story, critic_model, client, iteration):
    print("\n正在生成评审与质量评估……")
    critic_prompt = (
        "你是文学评论家。\n"
        "首先以 1–10 分评价故事（10 分为优秀）。\n"
        "然后分析所提供故事的优点与不足。\n"
        "评论情节、人物和整体感染力。\n"
        "如果需要改进，请提供 2–3 条具体建议。\n"
        "按照以下格式回复（保留英文标签）：\n"
        "Quality Score: [1-10]\n"
        "Critique: [详细评审]\n"
        "Suggestions: [分数低于 8 时的具体建议]\n\n"
        f"故事：\n{current_story}"
    )
    critic_response = client.models.generate_content(model=critic_model, contents=critic_prompt)
    critique_text = critic_response.text.strip()

    # 显示评审意见
    print(f"\n=== 评审 {iteration} ===\n")
    print(critique_text)

    # 从评审意见中提取质量分数
    try:
        # 尝试从评审意见中提取质量分数
        quality_line = [line for line in critique_text.split("\n") if "Quality Score:" in line]
        if quality_line:
            story_quality = int(quality_line[0].split(":")[1].strip().split()[0])
        else:
            story_quality = 0  # 未找到时使用默认值
    except (ValueError, IndexError):
        story_quality = 0  # 解析失败时使用默认值

    print(f"\n质量分数： {story_quality}/10")

    return critique_text, story_quality


@tracer.chain()
def revise_story(current_story, critique_text, story_model, client, iteration):
    print("\n正在生成修改稿……")
    revision_prompt = (
        "你是富有创意的短篇小说作家。\n"
        "仅根据给出的评审修改下方故事。\n"
        "重点处理评审中提到的具体建议。\n"
        "不要引入评审未要求的重要新元素。\n"
        "作出实质改进，提高质量分数。\n\n"
        f"Critique:\n{critique_text}\n\n"
        f"原始故事：\n{current_story}\n\n"
        "请根据建议生成改进后的故事。\n"
        "仅输出修改后的故事。"
    )
    revision_response = client.models.generate_content(model=story_model, contents=revision_prompt)
    revised_story = revision_response.text.strip()

    # 显示本轮改进后的故事
    print(f"\n=== 改进后的故事（迭代 {iteration}) ===\n")
    print(revised_story)

    return revised_story


generate_story(story_model, client)